In [14]:
import sys
from pathlib import Path

In [15]:
src_path = Path("../src").resolve()
sys.path.append(str(src_path))

In [17]:
from sqlmodel import Session,select
from api.events.models import EventModel
from api.db.session import engine
from api.db.config import DATABASE_URL
print(DATABASE_URL)
print(engine)

postgresql+psycopg://time-user:time-pw@db_service:5432/timescaledb
Engine(postgresql+psycopg://time-user:***@db_service:5432/timescaledb)


In [18]:
import pprint

try:
    with Session(engine) as session:
        print("s: ", session)
        print("e: ", engine)
        query = select(EventModel).order_by(EventModel.updated_at.desc()).limit(5)
        compile_query = query.compile(compile_kwargs ={"literal_binds":True})
        print(compile_query)
        print(str(query))
        results = session.exec(query).fetchall()
        print(results)
except Exception as e:
    pprint(e)

s:  <sqlmodel.orm.session.Session object at 0x00000144DC4CC550>
e:  Engine(postgresql+psycopg://time-user:***@db_service:5432/timescaledb)
SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at DESC
 LIMIT 5
SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at DESC
 LIMIT :param_1


TypeError: 'module' object is not callable. Did you mean: 'pprint.pprint(...)'?

In [6]:
from timescaledb.hyperfunctions import time_bucket
import pprint
with Session(engine) as session:
    bucket = time_bucket("1 day", EventModel.time)
    query = (
        select(bucket , EventModel.page)
            
        ).order_by(EventModel.updated_at.asc())
    
    compile_query = query.compile(compile_kwargs ={"literal_binds":True})
    results = session.exec(query).fetchall()
    pprint(results)

OperationalError: (psycopg.OperationalError) [Errno 11001] getaddrinfo failed
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# Test database connection
from sqlmodel import Session
from sqlalchemy import text
try:
    with Session(engine) as session:
        result = session.exec(text("SELECT 1")).first()
        print("Connection successful, result:", result)
except Exception as e:
    print("Connection failed:", e)

Connection failed: Textual SQL expression 'SELECT 1' should be explicitly declared as text('SELECT 1')
